# Oracle Hybrid → MonopolyZero behavioral clone

1. Upload **soft** `labels_hybrid_merged.npz` from coverage labeling (all incoming trades + routine subsample; keep paired `.jsonl`, no one-hot-only friend dumps).
2. Train soft-visit policy CE + backed-up value CE (`--weighted-sample` on by default; prefer resampled ACCEPT mix).
3. Validate: policy match on labels + 48-game seat-balanced H2H vs ASU.

**Gate:** expect high-30s to high-40s win rate vs ASU (oracle was ~53%). If mid-20s or lower, stop — do not start self-play.

**Label recipe (branch `oracle-hybrid-distill`):** buy/build/auction events + every incoming accept/decline once/round + `routine_label_prob=0.08` non-event Max-N labels. Regenerate ~30k soft rows before trusting BC.

In [ ]:
from pathlib import Path
import json, os, re, shutil, subprocess, sys, tarfile, urllib.request

CONTENT = Path(os.environ.get('MONOPOLYZERO_CONTENT', '/content'))
JOB = {
    'commit': '8ecbe4b52af2397688e852015d2e88d5e14e4677',
    'updates': 20000,
    'batch_size': 256,
    'eval_games': 48,
    'device': 'auto',
}
job_path = CONTENT / 'oracle-hybrid-bc-job.json'
if job_path.exists():
    JOB.update(json.loads(job_path.read_text()))
assert re.fullmatch(r'[0-9a-f]{40}', JOB['commit']), JOB['commit']
RUN_DIR = CONTENT / 'oracle-hybrid-bc-run'
STATUS_PATH = CONTENT / 'oracle-hybrid-bc-status.json'
EXAMPLES = CONTENT / 'labels_hybrid_merged.npz'
print(json.dumps(JOB, indent=2, sort_keys=True))

In [ ]:
import torch
STATUS = {
    **JOB,
    'state': 'setup',
    'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'torch': torch.__version__,
}
STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')
print(json.dumps(STATUS, indent=2))

In [ ]:
repository_archive = CONTENT / 'DeepRL_Monopoly.tar.gz'
if repository_archive.exists():
    with tarfile.open(repository_archive, 'r:gz') as archive:
        archive.extractall(CONTENT, filter='data')
    REPOSITORY_ROOT = CONTENT / 'DeepRL_Monopoly'
else:
    repository_archive = CONTENT / f"DeepRL_Monopoly-{JOB['commit']}.tar.gz"
    urllib.request.urlretrieve(
        f"https://codeload.github.com/ToprakG/DeepRL_Monopoly/tar.gz/{JOB['commit']}",
        repository_archive,
    )
    with tarfile.open(repository_archive, 'r:gz') as archive:
        roots = {Path(member.name).parts[0] for member in archive.getmembers() if member.name}
        assert len(roots) == 1
        archive.extractall(CONTENT, filter='data')
    REPOSITORY_ROOT = CONTENT / roots.pop()
assert (REPOSITORY_ROOT / 'oracle' / 'hybrid_distill.py').is_file()
baseline_bundle = CONTENT / 'monopolyzero-baselines.tar.gz'
if baseline_bundle.exists():
    with tarfile.open(baseline_bundle, 'r:gz') as archive:
        archive.extractall(REPOSITORY_ROOT, filter='data')
ppo = REPOSITORY_ROOT / 'artifacts/ppo_plus/ppo_hybrid_2000_v2.pt'
assert ppo.is_file(), 'Missing PPO bootstrap weights'
assert EXAMPLES.is_file(), f'Upload {EXAMPLES}'
sys.path.insert(0, str(REPOSITORY_ROOT))
print('repo', REPOSITORY_ROOT)
print('examples', EXAMPLES, 'bytes', EXAMPLES.stat().st_size)

In [ ]:
STATUS['state'] = 'training'
STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')
cmd = [
    sys.executable, '-m', 'oracle.hybrid_distill',
    '--examples', str(EXAMPLES),
    '--run-dir', str(RUN_DIR),
    '--bootstrap-ppo', str(ppo),
    '--updates', str(JOB['updates']),
    '--batch-size', str(JOB['batch_size']),
    '--device', str(JOB['device']),
    '--eval-games', str(JOB['eval_games']),
]
print(' '.join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=str(REPOSITORY_ROOT), check=False)
report_path = RUN_DIR / 'reports' / 'hybrid_bootstrap.json'
report = json.loads(report_path.read_text()) if report_path.exists() else {}
STATUS.update({
    'state': 'done' if proc.returncode == 0 else 'failed',
    'returncode': proc.returncode,
    'report': report,
})
STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')
print(json.dumps(report, indent=2, sort_keys=True))
if proc.returncode != 0:
    raise SystemExit(proc.returncode)
wr = report.get('h2h_vs_asu', {}).get('candidate_win_rate')
if wr is not None and wr < 0.28:
    raise RuntimeError(f'Clone vs ASU win_rate={wr:.3f} below 28% gate — do not self-play')
print('OK — warm start looks usable' if wr is None or wr >= 0.35 else 'MARGINAL — inspect before self-play')

In [ ]:
import tarfile
out = CONTENT / 'oracle-hybrid-bc-result.tar.gz'
with tarfile.open(out, 'w:gz') as archive:
    archive.add(RUN_DIR, arcname='run')
    archive.add(STATUS_PATH, arcname=STATUS_PATH.name)
print('wrote', out, out.stat().st_size)